# Certilab Adaptive RAG — Demo en vivo

Este notebook ejecuta el grafo **Adaptive RAG** canónico con 7 nodos y 2 loops
de auto-corrección, usando LangGraph + OpenAI + Tavily.

**Referencia**: [Building an Adaptive RAG System with LangGraph, OpenAI and Tavily](https://levelup.gitconnected.com/building-an-adaptive-rag-system-with-langgraph-openai-and-tavily-c4ee39d2f021)

### Requisitos
- `OPENAI_API_KEY` configurada en `.env` o exportada
- `TAVILY_API_KEY` opcional (activa la ruta de web search)
- Qdrant corriendo con datos indexados (`docker compose up -d qdrant` + `APP_MODE=real python -m app.adaptive_rag.ingest`)


In [ ]:
import os, sys
from dotenv import load_dotenv
load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    print("❌ Configura OPENAI_API_KEY en .env o exportala antes de ejecutar este notebook")
    sys.exit(1)
print("✅ OPENAI_API_KEY configurada")
print(f"   TAVILY_API_KEY: {'✅' if os.getenv('TAVILY_API_KEY') else '⚠️ no configurada (sin web search)'}")


In [ ]:
from pathlib import Path
from app.adaptive_rag.graph import build_graph
from app.adaptive_rag.grader import RouteQuery, GradeDocuments, GradeHallucinations, GradeAnswer
from app.adaptive_rag.state import AdaptiveRAGState
from app.config import Settings
from app.domain.models import Role
from app.security.access_control import Principal, scope_from_principal
from app.tools.web_search import TavilyWebSearch, WebSearchConfig

# Modo real: Qdrant con datos indexados
from app.retrieval.qdrant_index import QdrantVectorIndex
from app.tools.embeddings import EmbeddingProviderConfig, EmbeddingsProvider

print("✅ Imports OK")


In [ ]:
settings = Settings()
print(f"Modo: {settings.app_mode}")

# Construir componentes reales (Qdrant + OpenAI embeddings)
embedding_provider = EmbeddingsProvider(EmbeddingProviderConfig.from_settings(settings))
index = QdrantVectorIndex.from_settings(settings, embedding_provider)

tavily_key = settings.tavily_api_key
web_search = TavilyWebSearch(WebSearchConfig(tavily_api_key=tavily_key))
print(f"✅ Qdrant conectado")
print(f"✅ OpenAI embeddings: {settings.openai_embedding_model}")
print(f"✅ Web search: {'Tavily' if tavily_key else 'desactivado'}")

# Construir el grafo
graph = build_graph(index=index, embeddings=embedding_provider, web_search=web_search, settings=settings)
print(f"✅ Grafo Adaptive RAG compilado")


In [ ]:
def make_state(question: str, customer_id: int | None = None) -> AdaptiveRAGState:
    """Crear estado inicial con tenant isolation opcional."""
    principal = Principal(role=Role.ADMIN, customer_id=customer_id, user_id=1)
    scope = scope_from_principal(principal)
    return {
        "question": question, "generation": "", "documents": [],
        "web_results": [], "route": "", "rewrite_count": 0,
        "regenerate_count": 0, "hallucination_verdict": "",
        "answer_verdict": "", "principal": principal, "scope": scope,
    }

def stream_graph(initial_state: AdaptiveRAGState):
    """Ejecutar el grafo y mostrar cada nodo."""
    for step in graph.stream(initial_state):
        for node_name, node_output in step.items():
            print(f"--- Node: {node_name} ---")
            if isinstance(node_output, dict):
                for key, value in node_output.items():
                    preview = repr(value)
                    if len(preview) > 200:
                        preview = preview[:197] + "..."
                    print(f"  {key}: {preview}")
            print()

print("✅ Helpers definidos")


## 🔍 Query A: Ruta vectorstore — Procedimiento de calibración

Pregunta sobre un estándar técnico. El router debe clasificar como `vectorstore`
porque el vectorstore contiene documentos sobre normas INDECOPI.


In [ ]:
question_A = "¿Qué procedimiento de calibración sigue la norma INDECOPI?"
state_A = make_state(question_A)
stream_graph(state_A)


## 🎯 Query B: Tenant isolation — Certificados de un cliente

Pregunta sobre un cliente específico. El grafo detecta "ALS PERU" y filtra
la búsqueda solo a documentos de ese cliente (`allowed_customer_ids` en Qdrant).


In [ ]:
question_B = "¿Qué certificados tiene ALS PERU?"
state_B = make_state(question_B)
stream_graph(state_B)


## 🔄 Self-Correction Loops

El Adaptive RAG tiene **dos loops de auto-corrección**:

1. **Rewrite loop**: si ningún documento pasa el grading → reescribe la pregunta → reintenta (máx 3)
2. **Regenerate loop**: si la respuesta alucina → regenera (máx 2). Si no es útil → reescribe

Las siguientes queries demuestran ambos loops en acción.


### 🔄 Loop 1: Reescritura de query

Pregunta deliberadamente ambigua. El retriever devuelve documentos,
el grader los encuentra irrelevantes → el grafo reescribe la pregunta
y vuelve a intentar. Observá cómo `rewrite_count` se incrementa.


In [ ]:
question_C = "Dame info de calibración"
state_C = make_state(question_C)
stream_graph(state_C)


### 📊 Query D: Datos de tablas — Medición específica

Pregunta sobre datos numéricos de medición. El grafo recupera chunks
de tablas de calibración (splitteadas para no exceder el límite de embedding).
Si la primera respuesta no es útil, el loop de auto-corrección la refina.


In [ ]:
question_D = "¿Cuál fue la temperatura máxima registrada a 105°C?"
state_D = make_state(question_D)
stream_graph(state_D)


### 🏷️ Query E: Metadatos — Búsqueda por fecha y tipo

Pregunta que aprovecha los metadata chunks enriquecidos (fechas en español,
estado, tipo de documento). El grafo recupera chunks de tipo `metadata`
que contienen "mayo 2026" y "Acreditado".


In [ ]:
question_E = "¿Qué certificados acreditados se emitieron en mayo 2026?"
state_E = make_state(question_E)
stream_graph(state_E)


## 📊 Visualización del grafo

El grafo completo con 7 nodos y los 2 loops de auto-corrección.


In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))


## ✅ Resumen

Este notebook demostró:

- ✅ **Enrutamiento adaptativo**: el LLM decide vectorstore vs web search según la pregunta
- ✅ **Tenant isolation**: búsquedas filtradas por cliente (`allowed_customer_ids` en Qdrant)
- ✅ **Gradeo de relevancia**: cada documento recuperado es evaluado con LLM + Pydantic
- ✅ **Reescritura de query**: preguntas ambiguas se refinan automáticamente (máx 3 intentos)
- ✅ **Verificación de alucinaciones**: la respuesta generada se verifica contra los documentos fuente
- ✅ **Datos reales**: 154 certificados de calibración con metadatos de MySQL + PDFs de S3
- ✅ **Pipeline de ingesta**: PyMuPDF + Camelot + Unstructured → OpenAI embeddings → Qdrant

### Stack completo
Python 3.11 | LangGraph | LangChain | OpenAI (GPT-4o-mini + text-embedding-3-small) |
Tavily | Pydantic v2 | Qdrant | PyMuPDF | Camelot | Unstructured | Phoenix
